# 🧾 2.2 Prompt Templates in LangChain

## 🎯 Learning Objectives

In this notebook, you'll learn how to create **dynamic, reusable prompts** using LangChain's template system:

1. **PromptTemplate Basics** - Create parameterized prompts with variables
2. **Few-Shot Prompting** - Improve LLM outputs with examples
3. **Chain-of-Thought Prompting** - Guide reasoning step-by-step
4. **Prompt Composition** - Build complex prompts from reusable parts
5. **Serialization** - Save and load prompts from files

## 💡 Why Use Prompt Templates?

Instead of hardcoding prompts like:
```python
SystemMessage(content="You are a helpful assistant that translates English to Spanish.")
```

You can make them dynamic:
```python
template = "You are a helpful assistant that translates {input_language} to {output_language}"
```

---

## Prerequisites
- `langchain >= 1.4.0` and `langchain-core >= 1.6.1` (this repo's `pyproject.toml` floors)
- `OPENAI_API_KEY` in your project `.env` if you run the model cells
- Notebook `2.1_LangChain_Inputs_and_Outputs`

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API Keys & Import Dependencies
# ============================================================================
# We use python-dotenv to securely load API keys from a .env file
# This is a best practice - never hardcode API keys in your notebooks!
# ============================================================================

from dotenv import load_dotenv
import os
import sys
import platform

# Load environment variables from .env file
load_dotenv()

# Add parent directory to path for importing helpers
sys.path.append(os.path.abspath("../.."))

# Import our LLM factory functions
# - get_groq_llm(): Creates a Groq-hosted LLM (fast inference with open-source models)
# - get_openai_llm(): Creates an OpenAI GPT model
# - get_databricks_llm(): Creates a Databricks-hosted LLM
from helpers.utils import get_groq_llm, get_openai_llm, get_databricks_llm

print("✅ Environment variables loaded successfully!")
print(f"📍 Running on: {platform.system()}")

# -----------------------------------------------------------------------------
# Initialize the LLM based on platform or preference
# The choice of LLM affects tool calling capabilities and speed
# -----------------------------------------------------------------------------
if sys.platform == "win32":
    # Windows: Use Groq for fast inference
    llm = get_groq_llm()
elif sys.platform == "darwin":
    # macOS: Use Databricks-hosted Gemini
    llm = get_databricks_llm("databricks-gpt-5-1")  
else:
    # Linux: Default to Groq
    llm = get_groq_llm()

# Print which LLM we're using
if hasattr(llm, 'model_name'):
    print(f"🤖 LLM initialized: {llm.model_name}")
elif hasattr(llm, 'model'):
    print(f"🤖 LLM initialized: {llm.model}")
else:
    print("🤖 LLM initialized successfully")

In [ ]:
# ============================================================================
# BASIC PROMPT TEMPLATE
# ============================================================================
# A PromptTemplate uses Python's str.format() syntax with {variable_name}
# Variables are replaced with actual values when you call .format()
# ============================================================================

TEMPLATE = """
You are a helpful assistant that translates the {input_language} to {output_language}
"""

print("Template with placeholders:")
print(TEMPLATE)

In [ ]:
# ============================================================================
# CREATING A PROMPT TEMPLATE
# ============================================================================
# Method 1: .from_template() - Auto-detects variables from the template string
# This is the quickest way when your template is straightforward
# ============================================================================

from langchain_core.prompts import ChatPromptTemplate,PromptTemplate

# Create template - variables are automatically extracted from {placeholders}
prompt_template = PromptTemplate.from_template(template=TEMPLATE)

# Format the template with actual values
formatted_prompt = prompt_template.format(input_language="english", output_language="german")

print("📝 Formatted Prompt:")
print(formatted_prompt)

### Method 2: Explicit Variable Declaration

Passing `input_variables` to the constructor provides **validation** - LangChain will raise an error if you forget a variable or have a typo.

In [ ]:
# ============================================================================
# METHOD 2: Explicit Variable Declaration (Recommended for Production)
# ============================================================================
# By specifying input_variables, you get:
# - Validation that all variables exist in the template
# - Clear documentation of required inputs
# - Early error detection for typos
# ============================================================================

prompt_template = PromptTemplate(
    template=TEMPLATE, 
    input_variables=["input_language", "output_language"]
)

# This will work
print(prompt_template.format(input_language="english", output_language="german"))

# Uncommenting below would raise an error (missing variable):
# prompt_template.format(input_language="english")  # KeyError!


---

## 📚 Few-Shot Prompting

**Few-shot prompting** means providing examples in your prompt to guide the LLM's behavior. This technique:
- Improves output consistency and quality
- Teaches the model your expected format
- Reduces ambiguity in complex tasks

### Use Case: Sentiment Analysis with Subject Extraction

In [ ]:
# ============================================================================
# ZERO-SHOT PROMPT: no examples, so the output format may vary
# ============================================================================
# This template asks the LLM to analyze sentiment WITHOUT providing examples
# It works, but the output format may be inconsistent
# ============================================================================

TEMPLATE_ZERO_SHOT = """
Interprete the text and evaluate the text.
sentiment: is the text in a positive, neutral or negative sentiment?
subject: What subject is the text about? Use exactly one word.

Format the output as JSON with the following keys:
sentiment
subject

text: {input}
"""

print("📋 Zero-shot template (no examples):")
print(TEMPLATE_ZERO_SHOT)

### Adding Examples to Improve Quality

To improve performance and consistency, we provide **examples** that show the model exactly what output format we expect. This is called **few-shot prompting**.

In [ ]:
# ============================================================================
# FEW-SHOT PROMPT: examples steer both answer and format
# ============================================================================
# This template includes 9 examples covering:
# - 3 different restaurants (BellaVista, SeoulSavor, MunichMeals)
# - 3 sentiment types each (positive, neutral, negative)
# 
# Benefits of few-shot:
# - Consistent output format
# - Model learns your classification criteria
# - Reduces hallucinations
# ============================================================================

TEMPLATE_FEW_SHOT = """
Interprete the text and evaluate the text.
sentiment: is the text in a positive, neutral or negative sentiment?
subject: What subject is the text about? Use exactly one word.

Format the output as JSON with the following keys:
sentiment
subject

text: {input}

Examples:
text: The BellaVista restaurant offers an exquisite dining experience. The flavors are rich and the presentation is impeccable.
sentiment: positive
subject: BellaVista

text: BellaVista restaurant was alright. The food was decent, but nothing stood out.
sentiment: neutral
subject: BellaVista

text: I was disappointed with BellaVista. The service was slow and the dishes lacked flavor.
sentiment: negative
subject: BellaVista

text: SeoulSavor offered the most authentic Korean flavors I've tasted outside of Seoul. The kimchi was perfectly fermented and spicy.
sentiment: positive
subject: SeoulSavor

text: SeoulSavor was okay. The bibimbap was good but the bulgogi was a bit too sweet for my taste.
sentiment: neutral
subject: SeoulSavor

text: I didn't enjoy my meal at SeoulSavor. The tteokbokki was too mushy and the service was not attentive.
sentiment: negative
subject: SeoulSavor

text: MunichMeals has the best bratwurst and sauerkraut I've tasted outside of Bavaria. Their beer garden ambiance is truly authentic.
sentiment: positive
subject: MunichMeals

text: MunichMeals was alright. The weisswurst was okay, but I've had better elsewhere.
sentiment: neutral
subject: MunichMeals

text: I was let down by MunichMeals. The potato salad lacked flavor and the staff seemed uninterested.
sentiment: negative
subject: MunichMeals
"""

print(f"✅ Few-shot template created with examples for 3 restaurants x 3 sentiments = 9 examples")

In [ ]:
# ============================================================================
# USING THE FEW-SHOT TEMPLATE
# ============================================================================

prompt_template = PromptTemplate(template=TEMPLATE_FEW_SHOT, input_variables=["input"])

# Format with a new review to analyze
formatted_prompt = prompt_template.format(input="The MunichDeals experience was just awesome!")

print("📝 Formatted prompt (truncated for display):")
print(formatted_prompt[:300])  # First 300 chars -- the examples block is long

### Using FewShotPromptTemplate (Modular Approach)

LangChain provides `FewShotPromptTemplate` for a more **structured and maintainable** way to manage examples:
- Examples are stored as a list of dictionaries
- Easy to add, remove, or modify examples
- Cleaner separation of concerns

In [ ]:
# ============================================================================
# FEWSHOTPROMPTTEMPLATE: examples as data, not baked into a string
# ============================================================================
# Instead of embedding examples in a string, store them as structured data
# This makes it easy to:
# - Add/remove examples programmatically
# - Load examples from a database or file
# - Dynamically select relevant examples
# ============================================================================

from langchain_core.prompts import ChatPromptTemplate,PromptTemplate,FewShotPromptTemplate

# Examples stored as a list of dictionaries
examples = [
    {
        "text": "The BellaVista restaurant offers an exquisite dining experience. The flavors are rich and the presentation is impeccable.",
        "response": "sentiment: positive\nsubject: BellaVista"
    },
    {
        "text": "BellaVista restaurant was alright. The food was decent, but nothing stood out.",
        "response": "sentiment: neutral\nsubject: BellaVista"
    },
    # Note: Additional examples can be added here...
]

print(f"✅ Loaded {len(examples)} examples")

In [ ]:
# ============================================================================
# DYNAMICALLY ADDING EXAMPLES
# ============================================================================
# You can easily add new examples at runtime
# ============================================================================

new_example = {
    "text": "SeoulSavor was okay. The bibimbap was good but the bulgogi was a bit too sweet for my taste.",
    "response": "sentiment: neutral\nsubject: SeoulSavor"
}
examples.append(new_example)

print(f"✅ Added new example. Total examples: {len(examples)}")

In [ ]:
# ============================================================================
# EXAMPLE TEMPLATE
# ============================================================================
# This template defines how each example will be formatted
# The FewShotPromptTemplate will apply this to each example in the list
# ============================================================================

example_prompt = PromptTemplate(
    input_variables=["text", "response"], 
    template="Text: {text}\n{response}"
)

# Preview how one example looks when formatted
print("📋 Example format preview:")
print(example_prompt.format(**examples[0]))

In [ ]:
# ============================================================================
# BUILDING THE FEWSHOTPROMPTTEMPLATE
# ============================================================================
# Components:
# - examples: List of example dictionaries
# - example_prompt: Template for formatting each example
# - suffix: Text that comes after all examples (contains the actual input)
# - input_variables: Variables in the suffix that need values
# ============================================================================

prompt = FewShotPromptTemplate(
    examples=examples,           # Our list of examples
    example_prompt=example_prompt,  # How to format each example
    suffix="text: {input}",      # The actual query (comes after examples)
    input_variables=["input"]    # Variables we need to provide
)

print("✅ FewShotPromptTemplate created successfully!")

In [ ]:
# ============================================================================
# VIEWING THE FINAL PROMPT
# ============================================================================

final_prompt = prompt.format(input="The MunichDeals experience was just awesome!")

print("📝 Complete Few-Shot Prompt:")
print("=" * 60)
print(final_prompt)
print("=" * 60)

---

## 🧠 Chain-of-Thought (CoT) Prompting

**Chain-of-Thought prompting** goes beyond few-shot by showing the model the **reasoning process**, not just the final answer.

| Technique | Shows | Example |
|-----------|-------|---------|
| Few-shot | Input → Output | "Great food!" → positive |
| Chain-of-Thought | Input → Reasoning → Output | "Great food!" → "expresses satisfaction" → positive |

CoT helps with:
- Complex reasoning tasks
- Reducing errors
- Making outputs more explainable

In [ ]:
# ============================================================================
# CHAIN-OF-THOUGHT TEMPLATE
# ============================================================================
# This template includes the REASONING behind each classification
# The Q&A format guides the model through the thought process
# ============================================================================

TEMPLATE_COT = """
Interprete the text and evaluate the text. Determine if the text has a positive, neutral, or negative sentiment. Also, identify the subject of the text in one word.

Format the output as JSON with the following keys:
sentiment
subject

text: {input}

Chain-of-Thought Prompts:
Let's start by evaluating a statement. Consider: "The BellaVista restaurant offers an exquisite dining experience. The flavors are rich and the presentation is impeccable." How does this make you feel about BellaVista?
 It sounds like a positive review for BellaVista.

Based on the positive nature of that statement, how would you format your response?
 {{ "sentiment": "positive", "subject": "BellaVista" }}

Now, think about this: "SeoulSavor was okay. The bibimbap was good but the bulgogi was a bit too sweet for my taste." Does this give a strong feeling either way?
 Not particularly. It seems like a mix of good and not-so-good elements, so it's neutral.

Given the neutral sentiment, how should this be presented?
 {{ "sentiment": "neutral", "subject": "SeoulSavor" }}

Lastly, ponder on this: "I was let down by MunichMeals. The potato salad lacked flavor and the staff seemed uninterested." What's the overall impression here?
 The statement is expressing disappointment and dissatisfaction.

And if you were to categorize this impression, what would it be?
 {{ "sentiment": "negative", "subject": "MunichMeals" }}
"""

print("✅ Chain-of-Thought template created!")
print("💡 Notice how each example shows the REASONING, not just the answer.")

---

## 🧩 Prompt Composition — building one prompt from reusable parts

For complex prompts, you can **compose smaller templates** into a larger one:

- **Reusability**: use the same introduction or examples across several prompts
- **Maintainability**: update one component without touching the others
- **Flexibility**: mix and match components for different use cases

> **LangChain 1.x**: this section used to use `PipelinePromptTemplate`. That class
> is **gone** — not moved to `langchain-classic`, not renamed. It is absent from
> the installed packages entirely, so `from langchain.prompts.pipeline import
> PipelinePromptTemplate` raises `ModuleNotFoundError` with nothing to repoint to.
>
> The replacement is simpler than the thing it replaces: `PromptTemplate` supports
> the `+` operator. Adding templates concatenates them **and merges their
> `input_variables` for you** — which was the one genuinely useful thing the old
> class did.
>
> **The one thing `+` does not do** is named-slot placement. The old class injected
> each component into a `{slot}` of a final template, so parts could land out of
> order. If you need that, keep a final template with placeholders and fill it
> yourself — which is the form the class's own deprecation notice recommended:
>
> ```python
> final = PromptTemplate.from_template("{intro}\n\n{body}")
> final.format(**{name: part.format(**vals) for name, part in components})
> ```

In [ ]:
# ============================================================================
# PROMPT COMPOSITION: BUILD A PROMPT FROM REUSABLE PARTS
# ============================================================================
# Break a complex prompt into components:
#   1. Introduction - the task description (static)
#   2. Example      - a chain-of-thought demonstration
#   3. Execution    - the actual input to process
#
# Was: PipelinePromptTemplate(final_prompt=..., pipeline_prompts=[...])
# That class no longer exists anywhere in LangChain 1.x. See the note above.
# ============================================================================

from langchain_core.prompts import PromptTemplate

# ---
# Component 1: Introduction (no variables -- it is the same every time)
introduction_prompt = PromptTemplate.from_template(
    "Interprete the text and evaluate the text. Determine if the text has a "
    "positive, neutral, or negative sentiment. Also, identify the subject of "
    "the text in one word.\n\n"
)

# ---
# Component 2: Example (chain-of-thought demonstration)
# Named `example_component_prompt`, not `example_prompt`: cell 14 already
# binds `example_prompt` for the FewShotPromptTemplate in cell 15, and
# shadowing it makes re-running cell 15 fail with a confusing KeyError.
example_component_prompt = PromptTemplate.from_template(
    "Chain-of-Thought Prompts:\n"
    "Let's start by evaluating a statement. Consider: \"{example_text}\". "
    "How does this make you feel about {example_subject}?\n"
    "Response: {example_evaluation}\n\n"
    "Based on the {example_sentiment} nature of that statement, how would you "
    "format your response?\n"
    "Response: {example_format}\n\n"
)

# ---
# Component 3: Execution (the actual query)
execution_prompt = PromptTemplate.from_template(
    'Now, execute this process for the text: "{input}".'
)

# ---
# Compose with `+`. The result is an ordinary PromptTemplate whose
# input_variables is the UNION of the components' -- computed for you, which is
# exactly what PipelinePromptTemplate.input_variables used to provide.
composed_prompt = introduction_prompt + example_component_prompt + execution_prompt

print("✅ Composed 3 components with `+`")
print(f"📋 Required input variables: {sorted(composed_prompt.input_variables)}")

In [ ]:
# ============================================================================
# USING THE COMPOSED PROMPT
# ============================================================================
# Identical call shape to the old pipeline_prompt.format(...) -- the leaf
# variables are the same, because composition only concatenated the templates.
# ============================================================================

formatted = composed_prompt.format(
    # Example component variables
    example_text=(
        "The BellaVista restaurant offers an exquisite dining experience. "
        "The flavors are rich and the presentation is impeccable."
    ),
    example_subject="BellaVista",
    example_evaluation="It sounds like a positive review for BellaVista.",
    example_sentiment="positive",
    example_format='{ "sentiment": "positive", "subject": "BellaVista" }',
    # Execution component variable
    input="The new restaurant downtown has bland dishes and the wait time is too long.",
)

print("📋 Composed prompt:")
print("=" * 60)
print(formatted)
print("=" * 60)

# ---
# The other half of what PipelinePromptTemplate was used for: pre-filling a
# component so callers do not have to supply it. `.partial()` does that, and
# drops the filled variables from input_variables.
partially_filled = composed_prompt.partial(
    example_text="Service was slow but the food was outstanding.",
    example_subject="the bistro",
    example_evaluation="Overall a positive review of the bistro.",
    example_sentiment="positive",
    example_format='{ "sentiment": "positive", "subject": "bistro" }',
)

print()
print(f"🔧 After .partial(), callers only supply: {partially_filled.input_variables}")

---

## 💾 Serializing Prompts (Save & Load)

You can **save prompts to files** and load them later. This is useful for:
- Version control of prompts
- Sharing prompts across projects
- A/B testing different prompts

> **LangChain 1.x serializes to JSON**, via `dumps` / `loads` from
> `langchain_core.load`. The older `.save()` accepted a `.yaml` path; there is no
> 1.x equivalent for that, so YAML is now something you write yourself — see the
> note at the end of the next cell.
>
> **Note:** a prompt composed with `+` is an ordinary `PromptTemplate`, so it
> saves and loads like any other. The old `PipelinePromptTemplate` could not be
> serialized at all — one more reason composition beats the dedicated class.

In [ ]:
# ============================================================================
# SERIALIZATION: SAVE A PROMPT
# ============================================================================
# A prompt is data. Serializing it lets you version it in git, share it across
# projects, and A/B test variants without touching code.
#
# In LangChain 1.x the supported path is `dumps` / `dumpd` from
# `langchain_core.load`. The older `prompt.save("...")` still runs but is
# deprecated -- it emits:
#     The method `BasePromptTemplate.save` was deprecated in langchain-core 1.2.21 and will be removed in 2.0.0.
# ============================================================================

import json
from pathlib import Path

from langchain_core.load import dumps

# Named `joke_prompt`, not `prompt`: cell 15 binds `prompt` to the
# FewShotPromptTemplate that cell 16 formats, and shadowing it makes a re-run
# of cell 16 quietly format the wrong object.
joke_prompt = PromptTemplate(
    input_variables=["input"], template="Tell me a joke about {input}"
)

# `dumps` returns a JSON string describing the object and its type.
serialized = dumps(joke_prompt)
Path("prompt.json").write_text(serialized, encoding="utf-8")

print("✅ Saved to prompt.json")
print(f"📋 {len(serialized)} chars; top-level keys: "
      f"{sorted(json.loads(serialized).keys())}")

# --- Note on YAML ---
# The old `.save()` accepted a .yaml path. `dumps` is JSON-only, so that half of
# the old API has no 1.x equivalent -- write YAML yourself from `dumpd(joke_prompt)`
# (a plain dict) if you need it.

In [ ]:
# ============================================================================
# SERIALIZATION: LOAD IT BACK
# ============================================================================
# `loads` is the counterpart to `dumps`. The old `load_prompt("...")` still
# runs but is deprecated -- it emits:
#     The function `load_prompt` was deprecated in LangChain 1.2.21 and will be removed in 2.0.0.
# ============================================================================

# Expect one warning on this cell -- it is not a mistake:
#     The function `loads` is in beta. It is actively being worked on, so the
#     API may change.
# So the 1.x replacement for a deprecated API is itself still stabilising.
# It is the supported path regardless; just do not be surprised by the notice.

from langchain_core.load import loads

# `allowed_objects` is a security boundary, not boilerplate. Deserializing
# rebuilds arbitrary LangChain objects from the file's contents, so you name the
# classes you are willing to construct. Omit it and LangChain warns you.
restored = loads(
    Path("prompt.json").read_text(encoding="utf-8"),
    allowed_objects=[PromptTemplate],
)

print(f"📋 Restored type: {type(restored).__name__}")
print(restored.format(input="chickens"))

In [ ]:
# ============================================================================
# SERIALIZATION: CONFIRM THE ROUND TRIP
# ============================================================================
# Serializing is only useful if what comes back behaves identically.
# ============================================================================

same = restored.format(input="cows") == joke_prompt.format(input="cows")

print(f"✅ Round trip preserves rendering: {same}")
print(f"📋 input_variables preserved: {restored.input_variables == joke_prompt.input_variables}")
print()
print(restored.format(input="cows"))

---
## 📝 Summary

### 1. Templates
- **Key point**: `PromptTemplate` turns a string with `{variables}` into a reusable object
- **Key point**: few-shot examples steer both the answer *and* its format

### 2. Structuring examples
- **Key point**: `FewShotPromptTemplate` keeps examples as data rather than baked into a string
- **Key point**: chain-of-thought shows the reasoning, not just the answer

### 3. Composition
- **Key point**: `intro + example + execution` builds one prompt from reusable parts and
  merges their `input_variables` for you
- **Key point**: this replaces `PipelinePromptTemplate`, which no longer exists in 1.x

### 4. Serialization
- **Key point**: `dumps` / `loads` from `langchain_core.load` are the 1.x path;
  `.save()` and `load_prompt()` still work but are deprecated (removal 2.0.0)
- **Key point**: pass `allowed_objects=` when loading — it is a security boundary,
  since deserializing reconstructs arbitrary objects from file contents

### Next Steps
- `2.3_LLM_vs_ChatModel.ipynb` — the same ideas for chat models and message lists
- `2.5_Output_Parser.ipynb` — structuring what comes *back* from the model